# Understanding the Category Definition Pipeline

This notebook walks through the entire pipeline step-by-step, with layman explanations for every stage.

**What this does in simple terms:**
1. Search the web for expert articles about a software category
2. Filter out junk/vendors/ads
3. Read (scrape) the good articles
4. Score them with AI for quality
5. Use the best ones to write a category definition page

**Think of it like:** Asking a research assistant to find the best analyst reports, throwing away the brochures, and writing a summary from only the credible sources.

## Step 1: Install Required Tools

**In plain English:** Before we start, we need to install some Python libraries (like downloading apps before you can use them).

- `python-dotenv`: Reads secret API keys from a `.env` file so we don't hardcode passwords
- `langchain` / `langchain-openai`: Framework for talking to OpenAI's GPT models
- `trafilatura`: A web scraping library that extracts clean article text from websites (strips ads, menus, etc.)

In [1]:
# Install the Python libraries we need
%pip install python-dotenv langchain langchain-openai trafilatura -q

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## Step 2: Load Secret API Keys

**In plain English:** We need two API keys (like passwords) to use external services:

1. **OpenAI API key** - to use GPT for scoring articles and writing the final summary
2. **Serper API key** - to use Google's search engine programmatically

These are stored in a `.env` file (never commit this to git!). The code below reads them.

In [2]:
# Load secret API keys from .env file
import os
from dotenv import load_dotenv

load_dotenv()  # This reads the .env file in this folder

# Get the keys - if missing, the code will error and tell you
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
OPENAI_MODEL = os.getenv("OPENAI_MODEL", "gpt-4o-mini")  # Default model if not specified
SERPER_API_KEY = os.getenv("SERPER_API_KEY")

# Sanity check: make sure keys exist
assert OPENAI_API_KEY, "OPENAI_API_KEY not found in .env - add it!"
assert SERPER_API_KEY, "SERPER_API_KEY not found in .env - add it!"

print(f"Using OpenAI model: {OPENAI_MODEL}")
print(f"OpenAI key ends with: ...{OPENAI_API_KEY[-6:]}")
print(f"Serper key ends with: ...{SERPER_API_KEY[-6:]}")

Using OpenAI model: gpt-4o-mini
OpenAI key ends with: ...i-lLAA
Serper key ends with: ...d053d8


## Step 3: Set Up a Smart Cache

**In plain English:** API calls cost money. If we run this notebook twice, we don't want to pay again for the same search results.

This cache:
- Saves search results to disk as JSON files
- Uses a 'fingerprint' (SHA-256 hash) of the query as the filename
- Next time the same query runs, it reads from disk instead of calling the API

**Analogy:** Like keeping a notebook of answers so you don't have to ask the same question twice.

In [3]:
# Import libraries for caching
import hashlib, json, pathlib

# Create a .cache/ folder in this directory
CACHE_DIR = pathlib.Path(".cache")
CACHE_DIR.mkdir(exist_ok=True)

def _cache_key(*parts) -> str:
    """Create a unique fingerprint from query parameters."""
    raw = json.dumps(parts, sort_keys=True, ensure_ascii=True)
    return hashlib.sha256(raw.encode()).hexdigest()

def cache_get(namespace: str, *key_parts):
    """Try to load a cached result. Returns None if not found."""
    h = _cache_key(*key_parts)
    p = CACHE_DIR / namespace / f"{h}.json"
    if p.exists():
        return json.loads(p.read_text(encoding="utf-8"))
    return None

def cache_set(namespace: str, value, *key_parts):
    """Save a result to the cache for future runs."""
    h = _cache_key(*key_parts)
    d = CACHE_DIR / namespace
    d.mkdir(exist_ok=True)  # Create subfolder if needed
    p = d / f"{h}.json"
    p.write_text(json.dumps(value, ensure_ascii=False, indent=1), encoding="utf-8")

def cache_stats():
    """Print how many cached items we have."""
    if not CACHE_DIR.exists():
        print("Cache: empty")
        return
    for ns in sorted(CACHE_DIR.iterdir()):
        if ns.is_dir():
            files = list(ns.glob("*.json"))
            print(f"  {ns.name}: {len(files)} cached items")

print(f"Cache directory: {CACHE_DIR.resolve()}")
cache_stats()

Cache directory: C:\Users\Abhishek A\Defining_Category\.cache
  llm_scoring: 47 cached items
  llm_synthesis: 3 cached items
  serper: 80 cached items


## Step 4: Define What We're Searching For

**In plain English:** Now we set up the 'search mission.' We need to define:

1. **The category** - What software category are we researching? (e.g., "Account-Based Marketing")
2. **Aliases** - Other names people use (e.g., "ABM", "ABX")
3. **Trusted sites** - Which websites have expert analyst content? Organized in tiers:
   - **Tier 1:** Major analysts (Gartner, Forrester, IDC) - the gold standard
   - **Tier 2:** Independent analysts - still credible, often more accessible
   - **Tier 3:** Trade publications - industry news sites
4. **Blocklist** - What to EXCLUDE (vendor sites, review platforms, ads)

**Analogy:** Like giving a librarian a list of preferred publishers and a list of publishers to avoid.

In [4]:
# ── 4a: What category are we researching? ──
TEST_CATEGORY = "Account-Based Marketing"

# How mature is this category? This affects how old articles can be.
# "emerging" = 12 months max, "evolving" = 24 months, "stable" = 36 months
CATEGORY_MATURITY = "evolving"

# All the different names people use for this category
CATEGORY_ALIASES = [
    "Account-Based Marketing",
    "ABM",
    "Account-Based Marketing Platforms",
    "ABM platforms",
    "Account-Based Everything",
    "ABX",
    "Account-Based Experience",
]

print(f"Researching category: {TEST_CATEGORY}")
print(f"Maturity level: {CATEGORY_MATURITY}")
print(f"Number of aliases: {len(CATEGORY_ALIASES)}")

Researching category: Account-Based Marketing
Maturity level: evolving
Number of aliases: 7


In [5]:
# ── 4b: Trusted analyst sites (who we WANT to hear from) ──

# Tier 1: The biggest, most respected analyst firms
TIER1_SITES = [
    "gartner.com",           # Gartner - the most famous tech analyst
    "blogs.gartner.com",      # Gartner's blog network
    "gartner.com/en/articles",# Gartner curated articles
    "forrester.com",          # Forrester - another top analyst firm
    "go.forrester.com",       # Forrester's blog portal
    "idc.com",                # IDC - market research giant
    "blogs.idc.com",          # IDC analyst blogs
]

# Tier 2: Independent analysts - smaller but often more open with content
TIER2_SITES = [
    "constellationr.com",     # Constellation Research
    "infotech.com",           # Info-Tech Research Group
    "isg-one.com",            # ISG / Ventana Research
    "gigaom.com",             # GigaOm
    "aragonresearch.com",     # Aragon Research
    "nucleusresearch.com",    # Nucleus Research
    "hfsresearch.com",        # HFS Research
    "everestgrp.com",        # Everest Group
    "dresneradvisory.com",    # Dresner Advisory
    "451research.com",        # S&P Global / 451 Research
    "spglobal.com/marketintelligence",
    "omdia.tech.informa.com", # Omdia
    "abiresearch.com",        # ABI Research
    "moorinsightsstrategy.com", # Moor Insights
    "pund-it.com",           # Pund-IT
    "enderlegroup.com",       # Enderle Group
    "jgoldassociates.com",    # J. Gold Associates
]

# Tier 2b: Domain specialists (deep expertise in narrow areas)
TIER2B_SITES = [
    "kuppingercole.com",       # Identity/security
    "barc.com",               # Data/analytics
    "frost.com",              # Broad tech coverage
    "enterprisemanagement.com",# EMA - IT operations
    "esg-global.com",         # Enterprise Strategy Group
    "tag-cyber.com",          # Cybersecurity
    "securosis.com",          # Security research
    "colemanparkes.com",      # Telecom/IT
    
]

print(f"Tier 1 sites (major analysts): {len(TIER1_SITES)}")
print(f"Tier 2 sites (independent analysts): {len(TIER2_SITES)}")
print(f"Tier 2b sites (domain specialists): {len(TIER2B_SITES)}")

Tier 1 sites (major analysts): 7
Tier 2 sites (independent analysts): 17
Tier 2b sites (domain specialists): 8


In [6]:
# ── 4c: Direct URLs we know are goldmines ──

# These are specific analyst author pages or known good articles
# We add them directly instead of searching for them
ANALYST_HUB_URLS = [
    "https://www.forrester.com/blogs/author/john_arnold/",
    "https://www.forrester.com/blogs/author/jessie_johnson/",
    "https://www.forrester.com/blogs/author/terry_flaherty/",
    "https://www.gartner.com/en/articles/the-account-based-everything-framework",
    "https://research.isg-one.com/analyst-perspectives/topic/intelligent-marketing",
    "https://chiefmartec.com/category/account-based-marketing/",
]

print(f"Direct hub URLs added: {len(ANALYST_HUB_URLS)}")

Direct hub URLs added: 6


In [7]:
# ── 4d: The BLOCKLIST (what we do NOT want) ──

# These URL patterns indicate low-quality or vendor-biased content
DROP_URL_PATTERNS = [
    "/software-reviews/",       # Review platforms (not analyst research)
    "/compare/",                # Comparison pages
    "/products/",               # Product pages
    "gartner.com/reviews",     # Gartner Peer Insights (user reviews)
    "gartner.com/en/digital-markets",  # Capterra/GetApp (paid listings)
    "g2.com", "trustradius.com", "capterra.com", "getapp.com",
    "sourceforge.net", "goodfirms.co", "crozdesk.com",
    "/sponsors/",              # Event sponsor pages
    "/event/",                 # Event pages
]

# These entire domains are excluded from Google search at the query level
SERPER_EXCLUDE_SITES = [
    "store.frost.com",        # Paywall store
    "my.idc.com",             # IDC paywalled docs
    "info.idc.com",           # IDC gated content
    "linkedin.com",           # Social platform (not analyst research)
    "youtube.com",            # Videos
    "wikipedia.org",          # General knowledge
    "optimizely.com",         # Vendor sites
    "salesforce.com",
    "adobe.com",
    "oracle.com",
    "demandbase.com",
    "cognism.com",
    "zoomforth.com",
    "factors.ai",
    "influ2.com",
    "mutinyhq.com",
    "marketone.com",
    "strategicabm.com",
    "clay.com",
    "hginsights.com",
    "xgrowth.com.au",
    "datalane.com",
]

# Build the Google exclusion string (appended to every search)
_exc_parts = ["-filetype:pdf"]  # Skip PDFs
_exc_parts += [f"-site:{s}" for s in SERPER_EXCLUDE_SITES]
SERPER_EXCLUSIONS = " ".join(_exc_parts)

# Currency: how old can articles be?
CURRENCY_THRESHOLDS = {"emerging": 12, "evolving": 24, "stable": 36}
MAX_SOURCE_AGE_MONTHS = CURRENCY_THRESHOLDS[CATEGORY_MATURITY]

print(f"Drop patterns: {len(DROP_URL_PATTERNS)}")
print(f"Excluded sites: {len(SERPER_EXCLUDE_SITES)}")
print(f"Max article age: {MAX_SOURCE_AGE_MONTHS} months")
print(f"\nExclusion string ({len(SERPER_EXCLUSIONS)} chars):")
print(SERPER_EXCLUSIONS)

Drop patterns: 14
Excluded sites: 22
Max article age: 24 months

Exclusion string (434 chars):
-filetype:pdf -site:store.frost.com -site:my.idc.com -site:info.idc.com -site:linkedin.com -site:youtube.com -site:wikipedia.org -site:optimizely.com -site:salesforce.com -site:adobe.com -site:oracle.com -site:demandbase.com -site:cognism.com -site:zoomforth.com -site:factors.ai -site:influ2.com -site:mutinyhq.com -site:marketone.com -site:strategicabm.com -site:clay.com -site:hginsights.com -site:xgrowth.com.au -site:datalane.com


## Step 5: Search the Web (Using Google via Serper)

**In plain English:** Now we actually search Google. But we do it in a very specific way:

1. We use the `site:` operator to ONLY search our trusted sites
2. We search each alias (ABM, Account-Based Marketing, etc.) separately
3. We start with Tier 1 (best sources), then Tier 2, etc.
4. We exclude the blocklisted sites from the search itself

**Analogy:** Instead of searching "best cars" on Google and getting ads, we search "site:consumerreports.org OR site:edmunds.com best cars" - so only those sites can appear.

**Why cache?** Each search costs money via Serper. If we run this cell again, cached results load instantly for free.

In [8]:
# Import libraries for web searching
import requests, time
from urllib.parse import urlparse

SEARCH_DELAY = 0.3  # Wait 0.3 seconds between API calls (be polite)

def serper_search(query: str, api_key: str, num: int = 10) -> list[dict]:
    """Call Serper.dev (Google Search API) and cache results."""
    # First, check if we already have this search cached
    cached = cache_get("serper", query, num, {})
    if cached is not None:
        return cached
    
    # If not cached, call the API
    payload = {"q": query, "num": num}
    resp = requests.post(
        "https://google.serper.dev/search",
        headers={"X-API-KEY": api_key, "Content-Type": "application/json"},
        json=payload,
        timeout=15,
    )
    resp.raise_for_status()  # Error if something went wrong
    results = resp.json().get("organic", [])  # 'organic' = regular search results
    
    # Save to cache for next time
    cache_set("serper", results, query, num, {})
    time.sleep(SEARCH_DELAY)
    return results

print("✓ Search function ready")

✓ Search function ready


In [9]:
# Helper: batch sites into groups for efficient searching

def batch_site_queries(sites: list[str], batch_size: int = 5) -> list[str]:
    """Group sites into batches of 'batch_size' for OR queries."""
    batches = []
    for i in range(0, len(sites), batch_size):
        chunk = sites[i : i + batch_size]
        # Create: (site:a.com OR site:b.com OR site:c.com)
        clause = " OR ".join(f"site:{s}" for s in chunk)
        batches.append(f"({clause})")
    return batches

# Test: show what a batch looks like
test_batches = batch_site_queries(["gartner.com", "forrester.com", "idc.com"], batch_size=2)
for b in test_batches:
    print(b)

(site:gartner.com OR site:forrester.com)
(site:idc.com)


In [10]:
# Helper: check if a URL is from an allowed site

def url_is_from_allowed_sites(url: str, allowed_sites: list[str]) -> bool:
    """STRICT check: only allow URLs from our explicitly listed sites."""
    hostname = urlparse(url).netloc.lower()
    
    for allowed in allowed_sites:
        allowed_lower = allowed.lower()
        # Exact match or subdomain match
        if hostname == allowed_lower or hostname.endswith('.' + allowed_lower):
            return True
    return False

def url_is_blocked(url: str) -> bool:
    """Check if URL matches any drop patterns."""
    url_lower = url.lower()
    for pattern in DROP_URL_PATTERNS:
        if pattern in url_lower:
            return True
    return False

print("✓ URL filter functions ready")

✓ URL filter functions ready


In [11]:
# The main search function - runs one 'pass' over a tier of sites

def run_search_pass(name: str, sites: list[str], aliases: list[str],
                    batch_size: int, seen: set, results: list,
                    num_per_query: int = 6):
    """
    Run a search pass over a tier of sites.
    
    name: label for this pass (e.g., 'Tier1-primary')
    sites: list of domains to search
    aliases: search terms to try
    seen: set of already-found URLs (deduplication)
    results: list to append new results to
    """
    batches = batch_site_queries(sites, batch_size=batch_size)
    queries_run = 0
    hits_added = 0
    blocked = 0
    cache_hits = 0
    
    for alias in aliases:
        for site_clause in batches:
            # Build the query: site restrictions + alias + exclusions
            query = f'{site_clause} "{alias}" {SERPER_EXCLUSIONS}'
            queries_run += 1
            
            try:
                # Check cache first
                was_cached = cache_get("serper", query, num_per_query, {}) is not None
                if was_cached:
                    cache_hits += 1
                
                hits = serper_search(query, SERPER_API_KEY, num=num_per_query)
                
                for h in hits:
                    url = h.get("link", "")
                    if not url or url in seen:
                        continue  # Skip empty or duplicate URLs
                    
                    # STRICT: Only keep URLs from explicitly allowed sites
                    if not url_is_from_allowed_sites(url, sites):
                        blocked += 1
                        continue
                    
                    # Also check drop patterns
                    if url_is_blocked(url):
                        blocked += 1
                        continue
                    
                    seen.add(url)
                    results.append({
                        "url": url,
                        "title": h.get("title", ""),
                        "snippet": h.get("snippet", ""),
                        "query_alias": alias,
                        "search_pass": name,
                    })
                    hits_added += 1
                    
            except Exception as e:
                print(f"    ✗ Query failed: {e}")
    
    cached_msg = f", {cache_hits} from cache" if cache_hits else ""
    print(f"  {name}: {queries_run} queries → {hits_added} URLs (blocked {blocked}{cached_msg})")
    return queries_run

print("✓ Search pass function ready")

✓ Search pass function ready


In [12]:
# ── 5a: Define which aliases to use for each tier ──

# Primary aliases: the most common terms
FOCUSED_ALIASES = [
    "Account-Based Marketing",
    "ABM platforms",
]

# Secondary aliases: more specific terms (only searched on Tier 1)
SECONDARY_ALIASES = [
    "Account-Based Marketing Platforms",
    "Account-Based Everything",
]

# Key Tier 2 sites most relevant to this category
KEY_TIER2_SITES = [
    "constellationr.com",
    "isg-one.com",
    "gigaom.com",
    "nucleusresearch.com",
    "aragonresearch.com",
]

print(f"Primary aliases: {FOCUSED_ALIASES}")
print(f"Secondary aliases: {SECONDARY_ALIASES}")
print(f"Key Tier 2 sites: {len(KEY_TIER2_SITES)}")

Primary aliases: ['Account-Based Marketing', 'ABM platforms']
Secondary aliases: ['Account-Based Marketing Platforms', 'Account-Based Everything']
Key Tier 2 sites: 5


In [13]:
# ── 5b: Run the actual searches! ──

# These will hold our results across all passes
seen_urls = set()
all_results = []
total_queries = 0

print("=" * 60)
print(f"SEARCHING: {TEST_CATEGORY}")
print("=" * 60)

# Pass 0: Add direct hub URLs we already know about
print("\n── Pass 0: Direct analyst hub URLs ──")
hub_added = 0
for hub_url in ANALYST_HUB_URLS:
    if hub_url not in seen_urls and not url_is_blocked(hub_url):
        seen_urls.add(hub_url)
        all_results.append({
            "url": hub_url,
            "title": f"[Hub] {hub_url.split('/')[-2]}",
            "snippet": "",
            "query_alias": TEST_CATEGORY,
            "search_pass": "AnalystHub",
        })
        hub_added += 1
print(f"  Added {hub_added} direct URLs")

# Pass 1: Search Tier 1 (best analysts) with primary aliases
print("\n── Pass 1: Tier 1 sites + primary aliases ──")
total_queries += run_search_pass(
    "Tier1-primary", TIER1_SITES, FOCUSED_ALIASES,
    batch_size=3, seen=seen_urls, results=all_results, num_per_query=6
)

# Pass 1b: Search Tier 1 again with secondary aliases
print("\n── Pass 1b: Tier 1 sites + secondary aliases ──")
total_queries += run_search_pass(
    "Tier1-secondary", TIER1_SITES, SECONDARY_ALIASES,
    batch_size=3, seen=seen_urls, results=all_results, num_per_query=6
)

# Pass 2: Search key Tier 2 sites with primary aliases only
print("\n── Pass 2: Key Tier 2 sites + primary aliases ──")
total_queries += run_search_pass(
    "Tier2-key", KEY_TIER2_SITES, FOCUSED_ALIASES,
    batch_size=5, seen=seen_urls, results=all_results, num_per_query=6
)

# Summary
print(f"\n{'=' * 60}")
print(f"TOTAL: {total_queries} queries → {len(all_results)} unique URLs")
print(f"{'=' * 60}")

# Show breakdown by pass
from collections import Counter
pass_counts = Counter(r["search_pass"] for r in all_results)
for pass_name, count in pass_counts.items():
    print(f"  {pass_name}: {count} URLs")

# Show all found URLs
print(f"\nAll {len(all_results)} results:")
for i, r in enumerate(all_results):
    print(f"  {i+1}. [{r['search_pass']}] {r['title'][:70]}")
    print(f"     {r['url']}")

cache_stats()

SEARCHING: Account-Based Marketing

── Pass 0: Direct analyst hub URLs ──
  Added 6 direct URLs

── Pass 1: Tier 1 sites + primary aliases ──
  Tier1-primary: 6 queries → 17 URLs (blocked 6)

── Pass 1b: Tier 1 sites + secondary aliases ──
  Tier1-secondary: 6 queries → 10 URLs (blocked 2)

── Pass 2: Key Tier 2 sites + primary aliases ──
  Tier2-key: 2 queries → 6 URLs (blocked 0)

TOTAL: 14 queries → 39 unique URLs
  AnalystHub: 6 URLs
  Tier1-primary: 17 URLs
  Tier1-secondary: 10 URLs
  Tier2-key: 6 URLs

All 39 results:
  1. [AnalystHub] [Hub] john_arnold
     https://www.forrester.com/blogs/author/john_arnold/
  2. [AnalystHub] [Hub] jessie_johnson
     https://www.forrester.com/blogs/author/jessie_johnson/
  3. [AnalystHub] [Hub] terry_flaherty
     https://www.forrester.com/blogs/author/terry_flaherty/
  4. [AnalystHub] [Hub] articles
     https://www.gartner.com/en/articles/the-account-based-everything-framework
  5. [AnalystHub] [Hub] topic
     https://research.isg-one.com/a

## Step 6: Scrape (Read) the Found Articles

**In plain English:** We found URLs, but we only have titles and snippets. Now we need to actually visit each page and extract the full article text.

We use a library called **Trafilatura** which:
- Downloads the webpage
- Strips out navigation menus, ads, sidebars, footers
- Extracts just the article title, author, date, and body text

**Analogy:** Like a robot that goes to a webpage, throws away everything except the actual article, and brings back the clean text.

**Two-stage filtering:**
1. **Pre-scrape filter:** BEFORE visiting the page, we check if the URL looks promising (skip /author/ pages, /category/ pages, etc.)
2. **Post-scrape filter:** AFTER visiting, we check if the content is good (old articles, too short, no author, etc.)

In [14]:
# Import Trafilatura for web scraping
import trafilatura
import time as _time
from datetime import datetime

SCRAPE_DELAY = 0.5  # Wait 0.5 seconds between page fetches (be polite to servers)

def extract_article(url: str) -> dict:
    """
    Download a webpage and extract clean article content.
    Returns a dict with: url, title, author, date, text, hostname
    """
    try:
        downloaded = trafilatura.fetch_url(url)
        if not downloaded:
            return {"_error": "fetch returned empty (403/timeout/JS-only)"}
        
        # Extract clean text (no comments, no tables)
        text = trafilatura.extract(
            downloaded,
            include_comments=False,
            include_tables=False,
            output_format="txt",
        )
        if not text:
            return {"_error": "extraction returned no text"}
        
        # Extract metadata (author, date, site name)
        meta = trafilatura.metadata.extract_metadata(downloaded)
        return {
            "url": url,
            "title": meta.title if meta else "",
            "author": meta.author if meta else "",
            "date": meta.date if meta else "",
            "text": text,
            "hostname": meta.sitename if meta else "",
        }
    except Exception as e:
        return {"_error": f"exception: {type(e).__name__}: {e}"}

# Helper: calculate article age in months from date string
def source_age_months(date_str: str) -> int | None:
    """Calculate how old an article is in months."""
    if not date_str:
        return None
    try:
        dt = datetime.strptime(date_str[:10], "%Y-%m-%d")
        delta = datetime.now() - dt
        return int(delta.days / 30.44)
    except (ValueError, TypeError):
        return None

print("✓ Scraping functions ready")

✓ Scraping functions ready


In [15]:
# ── 6a: PRE-SCRAPE FILTER ──
# Decide whether to even visit a URL based on its pattern

def should_scrape_url(url: str, title: str, search_pass: str) -> tuple[bool, str]:
    """
    Pre-scrape filter: should we bother visiting this URL?
    Returns (True, "PASS") if yes, (False, reason) if no.
    """
    url_lower = url.lower()
    
    # 1. Check against drop patterns
    for pattern in DROP_URL_PATTERNS:
        if pattern in url_lower:
            return False, f"DROP pattern: {pattern}"
    
    # 2. Check blocked domains
    hostname = urlparse(url).netloc.lower()
    for blocked in SERPER_EXCLUDE_SITES:
        if blocked in hostname:
            return False, f"BLOCKED domain: {blocked}"
    
    # 3. Skip non-content pages by URL pattern
    non_content = [
        "/page/", "/category/", "/tag/", "/author/",
        "/search", "/login", "/register", "/contact",
        "/privacy", "/terms", "/about", "/careers",
        "/events", "/webinars", "/podcasts", "/videos",
        "/download", "/pdf", "/whitepaper", "/ebook",
        "/trial", "/demo", "/pricing", "/buy",
    ]
    for pattern in non_content:
        if pattern in url_lower:
            return False, f"NON_CONTENT: {pattern}"
    
    # 4. Skip if title is missing or too short
    if not title or len(title.strip()) < 10:
        return False, "SHORT_TITLE"
    
    # 5. Skip vendor marketing pages by title
    vendor_patterns = ["best", "top", "vs", "comparison", "review",
                       "pricing", "trial", "demo", "free"]
    title_lower = title.lower()
    for pattern in vendor_patterns:
        if pattern in title_lower and len(title_lower.split()) < 8:
            return False, f"VENDOR_TITLE: {pattern}"
    
    return True, "PASS"

# Apply the filter to all search results
filtered_for_scraping = []
pre_scrape_filtered = []

for result in all_results:
    should_scrape, reason = should_scrape_url(
        result["url"], result["title"], result["search_pass"]
    )
    if should_scrape:
        filtered_for_scraping.append(result)
    else:
        pre_scrape_filtered.append({
            "url": result["url"],
            "title": result["title"],
            "reason": reason,
        })

print(f"Original URLs: {len(all_results)}")
print(f"Pre-scrape filtered OUT: {len(pre_scrape_filtered)}")
print(f"Remaining to scrape: {len(filtered_for_scraping)}")

if pre_scrape_filtered:
    print("\nExamples filtered out:")
    for f in pre_scrape_filtered[:5]:
        print(f"  ✗ {f['reason']}: {f['title'][:50]}")

Original URLs: 39
Pre-scrape filtered OUT: 6
Remaining to scrape: 33

Examples filtered out:
  ✗ NON_CONTENT: /author/: [Hub] john_arnold
  ✗ NON_CONTENT: /author/: [Hub] jessie_johnson
  ✗ NON_CONTENT: /author/: [Hub] terry_flaherty
  ✗ VENDOR_TITLE: top: [Hub] topic
  ✗ NON_CONTENT: /category/: [Hub] account-based-marketing


In [16]:
# ── 6b: POST-SCRAPE FILTER ──
# After scraping, decide if the content is good enough to keep

def should_keep_scraped_content(article: dict, max_age_months: int) -> tuple[bool, str]:
    """
    Post-scrape filter: is this scraped content worth keeping?
    Returns (True, "PASS") if yes, (False, reason) if no.
    """
    # 1. Must have extracted successfully
    if not article or "_error" in article:
        return False, f"SCRAPE_ERROR: {article.get('_error', 'unknown')}"
    
    # 2. Must have substantial text
    text = article.get("text", "")
    if not text or len(text) < 300:
        return False, f"TOO_SHORT: {len(text)} chars"
    
    # 3. Check article age
    date_str = article.get("date")
    if date_str:
        age = source_age_months(date_str)
        if age is not None and age > max_age_months:
            return False, f"TOO_OLD: {age} months (max: {max_age_months})"
    
    # 4. Skip heavy marketing fluff
    marketing_phrases = [
        "best in class", "industry leading", "cutting edge",
        "revolutionary", "game changing", "breakthrough",
        "free trial", "contact us today", "get started now",
    ]
    text_lower = text.lower()
    marketing_count = sum(1 for p in marketing_phrases if p in text_lower)
    if marketing_count > 3 and len(text) < 1000:
        return False, f"TOO_MARKETING: {marketing_count} phrases"
    
    # 5. For analyst sites, expect a named author
    author = article.get("author", "")
    hostname = article.get("hostname", "")
    analyst_sites = ["gartner", "forrester", "idc", "constellation", "isg-one"]
    if any(a in hostname for a in analyst_sites):
        if not author or len(author.strip()) < 3:
            return False, "ANALYST_NO_AUTHOR"
    
    return True, "PASS"

print("✓ Post-scrape filter ready")

✓ Post-scrape filter ready


In [17]:
# ── 6c: Actually scrape the filtered URLs ──

scraped_sources = []
scrape_failures = []

print(f"Scraping {len(filtered_for_scraping)} URLs...")
print("=" * 60)

for i, result in enumerate(filtered_for_scraping):
    print(f"[{i+1}/{len(filtered_for_scraping)}] {result['url'][:70]}…", end=" ")
    
    # Visit the page and extract content
    article = extract_article(result["url"])
    
    # Apply post-scrape quality filter
    should_keep, keep_reason = should_keep_scraped_content(
        article, MAX_SOURCE_AGE_MONTHS
    )
    
    if should_keep:
        # Attach the search metadata to the article
        article["query_alias"] = result["query_alias"]
        article["search_pass"] = result["search_pass"]
        scraped_sources.append(article)
        print(f"✓ ({len(article['text'])} chars)")
    else:
        scrape_failures.append({
            "url": result["url"],
            "reason": keep_reason
        })
        print(f"✗ ({keep_reason})")
    
    _time.sleep(SCRAPE_DELAY)

# Print summary
print(f"\n{'='*60}")
print(f"SCRAPING RESULTS:")
print(f"  Original URLs: {len(all_results)}")
print(f"  Pre-scrape filtered: {len(pre_scrape_filtered)}")
print(f"  Attempted to scrape: {len(filtered_for_scraping)}")
print(f"  Successfully scraped: {len(scraped_sources)}")
print(f"  Post-scrape filtered: {len(scrape_failures)}")
print(f"  Overall rate: {len(scraped_sources)/len(all_results)*100:.1f}%")

Scraping 33 URLs...
[1/33] https://www.gartner.com/en/articles/the-account-based-everything-frame… ✗ (SCRAPE_ERROR: fetch returned empty (403/timeout/JS-only))
[2/33] https://www.gartner.com/en/documents/7152730… ✗ (SCRAPE_ERROR: fetch returned empty (403/timeout/JS-only))
[3/33] https://www.gartner.com/en/documents/7200530… ✗ (SCRAPE_ERROR: fetch returned empty (403/timeout/JS-only))
[4/33] https://www.gartner.com/en/documents/6479639… ✗ (SCRAPE_ERROR: fetch returned empty (403/timeout/JS-only))
[5/33] https://www.forrester.com/blogs/what-is-account-based-marketing/… ✗ (TOO_OLD: 96 months (max: 24))
[6/33] https://www.idc.com/resource-center/blog/how-abm-advertising-accelerat… ✓ (7888 chars)
[7/33] https://www.forrester.com/blogs/five-questions-to-ask-before-committin… ✗ (TOO_OLD: 35 months (max: 24))
[8/33] https://www.forrester.com/blogs/accountbased-marketing-four-operationa… ✗ (TOO_OLD: 156 months (max: 24))
[9/33] https://www.forrester.com/report/the-future-of-account-based-marke

## Step 7: Remove Duplicate Content

**In plain English:** Sometimes different URLs return the exact same article (e.g., a comparison table copied across multiple pages). We don't want to process the same text twice.

We create a 'fingerprint' (hash) of the first 500 characters of each article. If two articles have the same fingerprint, we consider them duplicates and keep only one.

**Analogy:** Like comparing the first paragraph of two books - if they're identical, it's probably the same book.

In [18]:
# Deduplicate scraped content by hashing first 500 chars
import hashlib

def _content_hash(text: str) -> str:
    """Create a fingerprint from the first 500 characters."""
    return hashlib.md5(text[:500].strip().lower().encode()).hexdigest()

_seen_hashes = set()
deduped_sources = []
dupes_removed = 0

for s in scraped_sources:
    h = _content_hash(s["text"])
    if h in _seen_hashes:
        dupes_removed += 1
        continue
    _seen_hashes.add(h)
    deduped_sources.append(s)

print(f"Before dedup: {len(scraped_sources)} sources")
print(f"Duplicates removed: {dupes_removed}")
print(f"After dedup: {len(deduped_sources)} unique sources")

# Replace scraped_sources with deduped list for downstream cells
scraped_sources = deduped_sources

Before dedup: 5 sources
Duplicates removed: 0
After dedup: 5 unique sources


## Step 8: Score Each Source with AI (LLM)

**In plain English:** Now we use GPT (the AI) to judge each article's quality. We send the article text to GPT and ask it to score on several criteria.

**What we score:**
1. **Slot-fill** - Does the article cover the 5 key topics? (definition, capabilities, boundaries, buyers, vendors)
2. **Function verbs** - Does it use expert language ("orchestrate", "unify") vs marketing fluff ("better", "smarter")?
3. **Byline quality** - Is the author a named analyst or anonymous?
4. **Vendor diversity** - Does it mention many vendors or just promote one?
5. **Relevance** - Overall score 1-10

**Analogy:** Like hiring an editor to read each article and grade it on a rubric.

**Why cache?** GPT calls cost money (tokens). We cache scores so we don't re-grade the same article twice.

In [ ]:
# Import LangChain for structured GPT output
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from pydantic import BaseModel, Field

LLM_DELAY = 0.2  # Wait between GPT calls

# Define the structured output format for GPT
class SourceScore(BaseModel):
    """The grades GPT will assign to each article."""
    slot_definition: bool = Field(description="Has a clear definition of the category")
    slot_capabilities: bool = Field(description="Lists core software capabilities")
    slot_boundaries: bool = Field(description="Explains what this is NOT")
    slot_buyer_use: bool = Field(description="Describes who buys it and why")
    slot_vendors: bool = Field(description="Names representative vendors")
    slots_filled: int = Field(description="How many of the 5 slots (0-5)")
    uses_function_verbs: bool = Field(description="Uses expert verbs, not marketing fluff")
    vendor_count: int = Field(description="Number of distinct vendors mentioned")
    vendor_names: list[str] = Field(description="Names of vendors found")
    is_sme_content: bool = Field(description="Is this expert content, not marketing?")
    byline_quality: str = Field(description="'named_analyst', 'named_author', or 'no_byline'")
    single_vendor_bias: bool = Field(description="Does it only promote one vendor?")
    relevance_score: int = Field(description="Overall relevance 1-10")
    reasoning: str = Field(description="Why you gave these scores")

print("✓ Score structure defined")

In [ ]:
# Define the prompt we send to GPT for grading

SCORING_PROMPT = ChatPromptTemplate.from_messages([
    ("system", """You are evaluating a web source for defining the software category "{category}".

Score it on these criteria:

1. SLOT-FILL: Does it have (a) definition, (b) capabilities, (c) boundaries, (d) buyers, (e) vendors?

2. FUNCTION-VERBS: Does it use expert verbs (orchestrate, unify, score, route, segment) rather than marketing adjectives (better, smarter, faster)?

3. BYLINE: Is the author a named analyst at a known firm, a named author, or anonymous?

4. VENDOR DIVERSITY: How many vendors are named? Does it promote just one vendor?

5. SME CONTENT: Is this expert analysis or marketing fluff?

6. RELEVANCE: Overall 1-10 score for usefulness in writing a category page.
   - 8-10: Fills 4-5 slots, expert verbs, named analyst, multi-vendor
   - 5-7: Fills 2-3 slots or partial quality
   - 1-4: Off-topic, thin, or promotional
"""),
    ("human", """Source URL: {url}
Title: {title}
Author: {author}
Date: {date}
Host: {hostname}

Content (first 3000 chars):
{text}

Return your evaluation as structured JSON."""),
])

# Create the GPT-powered scoring chain
llm = ChatOpenAI(model=OPENAI_MODEL, temperature=0, api_key=OPENAI_API_KEY)
structured_llm = llm.with_structured_output(SourceScore)
chain = SCORING_PROMPT | structured_llm

print(f"✓ Scoring chain ready (model: {OPENAI_MODEL})")

In [ ]:
# ── 8c: Run the scoring on all scraped articles ──

scored_sources = []
cache_hits_llm = 0

print(f"Scoring {len(scraped_sources)} sources with GPT...")
print("=" * 60)

for i, s in enumerate(scraped_sources):
    print(f"[{i+1}/{len(scraped_sources)}] Scoring: {s['title'][:50]}…", end=" ")
    
    # Prepare inputs
    text_trunc = s["text"][:3000]
    age = source_age_months(s.get("date"))
    age_str = f"{age} months" if age is not None else "unknown"
    
    invoke_params = {
        "category": TEST_CATEGORY,
        "url": s["url"],
        "title": s["title"] or "Unknown",
        "author": s["author"] or "Unknown",
        "date": s["date"] or "Unknown",
        "hostname": s["hostname"] or "Unknown",
        "source_age": age_str,
        "text": text_trunc,
    }
    
    # Build cache key from all inputs
    cache_key_parts = ("scoring_v2", TEST_CATEGORY, s["url"],
                       s["title"] or "Unknown", s["author"] or "Unknown",
                       s["date"] or "Unknown", s["hostname"] or "Unknown",
                       text_trunc)
    
    # Check cache first
    cached = cache_get("llm_scoring", *cache_key_parts)
    if cached is not None:
        score_data = cached["score"] if "score" in cached else cached
        score = SourceScore(**score_data)
        cache_hits_llm += 1
        scored_sources.append({"source": s, "score": score})
        print(f"✓ relevance={score.relevance_score}/10 (cached)")
        continue
    
    # Call GPT if not cached
    try:
        score = chain.invoke(invoke_params)
        
        # Cache the result
        cache_set("llm_scoring", {
            "score": score.model_dump(),
        }, *cache_key_parts)
        
        scored_sources.append({"source": s, "score": score})
        print(f"✓ relevance={score.relevance_score}/10")
        _time.sleep(LLM_DELAY)
        
    except Exception as e:
        print(f"✗ {e}")

# Sort by relevance (best first)
scored_sources.sort(key=lambda x: x["score"].relevance_score, reverse=True)

print(f"\nScored {len(scored_sources)} sources.")
if cache_hits_llm:
    print(f"  ({cache_hits_llm} from cache, {len(scored_sources) - cache_hits_llm} new GPT calls)")
cache_stats()

## Step 9: See the Scoring Results

**In plain English:** Now we can see which articles GPT thinks are the best. We display them sorted by relevance score (highest first).

We also filter to only keep articles that:
- Score 5 or higher (decent quality)
- Fill at least 2 of 5 slots (covers enough topics)
- Don't have single-vendor bias (not just a sales pitch for one company)

In [ ]:
# Display all scored sources, ranked best to worst

for i, item in enumerate(scored_sources):
    s = item["source"]
    sc = item["score"]
    print(f"{'='*60}")
    print(f"#{i+1}  Relevance: {sc.relevance_score}/10 | Slots: {sc.slots_filled}/5 | Vendors: {sc.vendor_count}")
    print(f"  Title : {s['title']}")
    print(f"  Author: {s['author'] or 'n/a'} | Date: {s['date'] or 'n/a'} | Host: {s['hostname'] or 'n/a'}")
    print(f"  URL   : {s['url']}")
    print(f"  Byline: {sc.byline_quality} | Expert verbs: {sc.uses_function_verbs} | SME: {sc.is_sme_content}")
    print(f"  Slots → Def:{sc.slot_definition} Cap:{sc.slot_capabilities} Bound:{sc.slot_boundaries} Buyer:{sc.slot_buyer_use} Vendors:{sc.slot_vendors}")
    if sc.vendor_names:
        print(f"  Vendors named: {', '.join(sc.vendor_names)}")
    print(f"  Reason: {sc.reasoning[:150]}…")
    print()

# Select top sources for final synthesis
top_sources = [
    item for item in scored_sources
    if item["score"].relevance_score >= 5
    and item["score"].slots_filled >= 2
    and not item["score"].single_vendor_bias
]

# If too few sources pass, relax the bias filter
if len(top_sources) < 3:
    top_sources = [
        item for item in scored_sources
        if item["score"].relevance_score >= 5
        and item["score"].slots_filled >= 2
    ]

print(f"\n{'='*60}")
print(f"Sources selected for synthesis: {len(top_sources)}")
print(f"  (relevance ≥ 5, slots ≥ 2, no single-vendor bias)")
if len(top_sources) < 3:
    print("⚠️  WARNING: Fewer than 3 sources — synthesis may be thin.")

## Step 10: Synthesize the Final Category Page

**In plain English:** Now for the grand finale! We take all the high-scoring articles and feed them to GPT one more time. We ask GPT to write a comprehensive category definition page by combining insights from ALL the sources.

**The output includes:**
1. **Definition** - What is this software category?
2. **Core capabilities** - What does the software DO?
3. **Boundaries** - What is it NOT? (adjacent categories)
4. **Buyer/use case** - Who buys it and why?
5. **Vendors** - Which companies make this software?
6. **Market overview** - Market size, growth, trends
7. **Implementation** - How do companies adopt it?
8. **Future trends** - Where is this category headed?

**Important rule:** GPT must NOT directly copy text from sources (copyright). It must synthesize and rewrite in its own words.

**Analogy:** Like a journalist interviewing 17 experts and writing an original article that captures the consensus view.

In [ ]:
# Define the structured output for the final category page

class CategoryPage(BaseModel):
    """The final synthesized category definition page."""
    category_name: str = Field(description="Primary category name")
    aliases: list[str] = Field(description="Other names for this category")
    definition: str = Field(description="2-4 sentence definition")
    core_capabilities: list[str] = Field(description="What the software does")
    boundaries: str = Field(description="What this is NOT")
    buyer_use_case: str = Field(description="Who buys it and why")
    representative_vendors: list[str] = Field(description="Key vendor names")
    category_drift: str = Field(description="Where analysts disagree")
    market_overview: str = Field(description="Market size and trends")
    implementation_considerations: str = Field(description="How to implement")
    vendor_landscape: str = Field(description="Competitive dynamics")
    future_trends: str = Field(description="Emerging directions")
    integration_points: str = Field(description="How it connects to other systems")
    success_metrics: str = Field(description="KPIs to measure success")
    common_challenges: str = Field(description="Typical hurdles")
    source_count: int = Field(description="How many sources were used")
    confidence: str = Field(description="high/medium/low based on source quality")

print("✓ Category page structure defined")

In [ ]:
# Define the synthesis prompt

SYNTHESIS_PROMPT = ChatPromptTemplate.from_messages([
    ("system", """You are writing a comprehensive category definition page.

RULES:
- Write in your own words — NO direct quotes from sources (copyright)
- Multi-source consensus carries the definition
- Use function-verbs (orchestrate, unify, score, route, segment) not adjectives (better, smarter)
- Be specific with vendor names, buyer roles, and examples
- If analysts disagree, name the specific firms and their positions

Category: {category}
Maturity: {maturity}
Aliases: {aliases}
"""),
    ("human", """Synthesize these sources into a comprehensive category page:

{sources_text}
"""),
])

print("✓ Synthesis prompt ready")

In [ ]:
# ── 10c: Prepare sources and run synthesis ──

if not top_sources:
    print("❌ No sources available for synthesis!")
else:
    print(f"Synthesizing from {len(top_sources)} high-quality sources...")
    print("=" * 60)
    
    # Build the sources text block for GPT
    sources_block = ""
    for i, item in enumerate(top_sources):
        s = item["source"]
        sc = item["score"]
        sources_block += f"\n--- SOURCE {i+1} ---\n"
        sources_block += f"Title: {s['title']}\n"
        sources_block += f"Author: {s['author'] or 'Unknown'} | Host: {s['hostname'] or 'Unknown'}\n"
        sources_block += f"Score: relevance={sc.relevance_score}/10, slots={sc.slots_filled}/5\n"
        sources_block += f"Content: {s['text'][:4000]}\n"
    
    # Check cache
    synth_cache_key = ("synthesis_v2", TEST_CATEGORY, ", ".join(CATEGORY_ALIASES), sources_block)
    cached_synth = cache_get("llm_synthesis", *synth_cache_key)
    
    if cached_synth is not None:
        category_page = CategoryPage(**cached_synth)
        print("✓ Synthesis loaded from cache!")
    else:
        # Create synthesis chain
        synth_llm = ChatOpenAI(model=OPENAI_MODEL, temperature=0.2, api_key=OPENAI_API_KEY)
        synth_chain = SYNTHESIS_PROMPT | synth_llm.with_structured_output(CategoryPage)
        
        # Run synthesis
        category_page = synth_chain.invoke({
            "category": TEST_CATEGORY,
            "maturity": CATEGORY_MATURITY,
            "aliases": ", ".join(CATEGORY_ALIASES),
            "sources_text": sources_block,
        })
        
        # Cache result
        cache_set("llm_synthesis", category_page.model_dump(), *synth_cache_key)
        print("✓ Synthesis complete (cached)!")
    
    print(f"\n{'='*60}")
    print(f"  CATEGORY PAGE: {category_page.category_name}")
    print(f"{'='*60}")
    print(f"\nAliases: {', '.join(category_page.aliases)}")
    print(f"Confidence: {category_page.confidence} | Sources: {category_page.source_count}")

## Step 11: Display the Final Output

**In plain English:** This is the result! A comprehensive category definition page generated from all the expert sources we found and scored.

Each section is a synthesis of multiple analyst sources - not copied from any single article.

In [ ]:
# Display the full synthesized category page

print(f"\n{'─' * 60}")
print("1. DEFINITION")
print(f"{'─' * 60}")
print(category_page.definition)

print(f"\n{'─' * 60}")
print("2. CORE CAPABILITIES")
print(f"{'─' * 60}")
for cap in category_page.core_capabilities:
    print(f"  • {cap}")

print(f"\n{'─' * 60}")
print("3. BOUNDARIES (What this is NOT)")
print(f"{'─' * 60}")
print(category_page.boundaries)

print(f"\n{'─' * 60}")
print("4. BUYER / USE CASE")
print(f"{'─' * 60}")
print(category_page.buyer_use_case)

print(f"\n{'─' * 60}")
print("5. REPRESENTATIVE VENDORS")
print(f"{'─' * 60}")
for v in category_page.representative_vendors:
    print(f"  • {v}")

print(f"\n{'─' * 60}")
print("6. MARKET OVERVIEW")
print(f"{'─' * 60}")
print(category_page.market_overview)

print(f"\n{'─' * 60}")
print("7. IMPLEMENTATION CONSIDERATIONS")
print(f"{'─' * 60}")
print(category_page.implementation_considerations)

print(f"\n{'─' * 60}")
print("8. VENDOR LANDSCAPE")
print(f"{'─' * 60}")
print(category_page.vendor_landscape)

print(f"\n{'─' * 60}")
print("9. FUTURE TRENDS")
print(f"{'─' * 60}")
print(category_page.future_trends)

print(f"\n{'─' * 60}")
print("10. INTEGRATION POINTS")
print(f"{'─' * 60}")
print(category_page.integration_points)

print(f"\n{'─' * 60}")
print("11. SUCCESS METRICS")
print(f"{'─' * 60}")
print(category_page.success_metrics)

print(f"\n{'─' * 60}")
print("12. COMMON CHALLENGES")
print(f"{'─' * 60}")
print(category_page.common_challenges)

if category_page.category_drift:
    print(f"\n{'─' * 60}")
    print("13. CATEGORY DRIFT / ANALYST DISAGREEMENT")
    print(f"{'─' * 60}")
    print(category_page.category_drift)

print(f"\n{'='*60}")
print("Sources used in synthesis:")
for i, item in enumerate(top_sources):
    s = item["source"]
    sc = item["score"]
    print(f"  [{i+1}] {s['title'][:60]} — {s['author'] or 'n/a'} ({s['hostname'] or 'n/a'})")
    print(f"      Relevance: {sc.relevance_score}/10, Slots: {sc.slots_filled}/5")

## Step 12: Export Results to Files

**In plain English:** Save everything to disk so we can use it later or share it.

We export:
1. The final category page as JSON
2. All search results as JSON
3. All scraped sources as JSON
4. All scores as JSON
5. A summary of the pipeline run

In [ ]:
# Export everything to the output/ folder
import json, pathlib
from datetime import datetime

output_dir = pathlib.Path("output")
output_dir.mkdir(exist_ok=True)

cat_slug = TEST_CATEGORY.lower().replace(" ", "_").replace("-", "_")

# 1. Export category page
if 'category_page' in locals() and category_page:
    page_path = output_dir / f"{cat_slug}_page.json"
    page_path.write_text(
        json.dumps(category_page.model_dump(), indent=2, ensure_ascii=False),
        encoding="utf-8"
    )
    print(f"✓ Category page: {page_path}")

# 2. Export all search results
links_path = output_dir / f"{cat_slug}_search_results.json"
links_path.write_text(
    json.dumps(all_results, indent=2, ensure_ascii=False),
    encoding="utf-8"
)
print(f"✓ Search results: {links_path}")

# 3. Export scraped sources
scraped_export = []
for s in scraped_sources:
    scraped_export.append({
        "url": s["url"],
        "title": s["title"],
        "author": s["author"],
        "date": s["date"],
        "hostname": s["hostname"],
        "search_pass": s.get("search_pass", ""),
        "text_length": len(s["text"]),
    })
scraped_path = output_dir / f"{cat_slug}_scraped.json"
scraped_path.write_text(
    json.dumps(scraped_export, indent=2, ensure_ascii=False),
    encoding="utf-8"
)
print(f"✓ Scraped sources: {scraped_path}")

# 4. Export scores
scores_export = []
for item in scored_sources:
    s = item["source"]
    sc = item["score"]
    scores_export.append({
        "url": s["url"],
        "title": s["title"],
        "author": s["author"],
        "score": sc.model_dump(),
    })
scores_path = output_dir / f"{cat_slug}_scores.json"
scores_path.write_text(
    json.dumps(scores_export, indent=2, ensure_ascii=False),
    encoding="utf-8"
)
print(f"✓ Scores: {scores_path}")

# 5. Summary
summary = {
    "category": TEST_CATEGORY,
    "timestamp": datetime.now().isoformat(),
    "stats": {
        "urls_found": len(all_results),
        "urls_scraped": len(scraped_sources),
        "sources_scored": len(scored_sources),
        "sources_synthesized": len(top_sources),
    }
}
summary_path = output_dir / f"{cat_slug}_summary.json"
summary_path.write_text(
    json.dumps(summary, indent=2, ensure_ascii=False),
    encoding="utf-8"
)
print(f"✓ Summary: {summary_path}")

print(f"\n🎉 All files exported to {output_dir}/")

## Pipeline Complete!

**What we just did (summary):**

| Step | What Happened | Why It Matters |
|------|--------------|----------------|
| 1-2 | Installed tools, loaded API keys | We need these to run |
| 3 | Set up cache | Saves money on re-runs |
| 4 | Defined category + trusted sites + blocklist | Controls what we search and exclude |
| 5 | Searched Google via Serper | Found candidate articles |
| 6 | Scraped articles with Trafilatura | Got clean text + metadata |
| 6a-b | Pre/post-scrape filters | Threw away junk before/after scraping |
| 7 | Deduplicated | Removed duplicate content |
| 8 | Scored with GPT | Graded each article on quality |
| 9 | Selected top sources | Only kept the best articles |
| 10 | Synthesized with GPT | Wrote the final category page |
| 11 | Displayed results | Showed the output |
| 12 | Exported to JSON | Saved everything to disk |

**Key design principles:**
- **Strict site filtering:** Only searched explicitly listed sites
- **Two-stage filtering:** Filter URLs before AND after scraping
- **Caching:** Save API calls for re-runs
- **Scoring rubric:** GPT grades on objective criteria, not gut feel
- **Synthesis:** Combine multiple sources, never copy directly